Improving Instruction-Data Via Reflection-Tuning Using GPT-4

In [ ]:
# 检查本 notebook 依赖的第三方库版本（openai 用于调用 GPT-4/GPT-4o-mini API，tqdm 用于显示进度条）
from importlib.metadata import version

pkgs = [
    "openai",  # OpenAI API
    "tqdm",    # Progress bar
]

# 遍历依赖包列表，打印各自已安装的版本号，便于复现环境
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 从本地 config.json 中读取 OpenAI API Key，并创建 OpenAI 客户端，用于后续调用 GPT-4/GPT-4o-mini 完成“反思式”数据改写
import json
from openai import OpenAI

# Load API key from a JSON file.
# Make sure to replace "sk-..." with your actual API key from https://platform.openai.com/api-keys
with open("config.json", "r") as config_file:
    config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]

# 用读取到的 api_key 初始化 OpenAI 客户端实例，后续所有 API 调用都通过它完成
client = OpenAI(api_key=api_key)

In [ ]:
# 封装一个通用的 ChatGPT 调用函数：给定用户 prompt（以及可选的 system prompt），返回模型的文本回复
def run_chatgpt(prompt, client, model="gpt-4o-mini", system_prompt=None):
    # Define the system message if a system_prompt is provided
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    # Add the user prompt to the messages
    messages.append({"role": "user", "content": prompt})

    # Call the API
    # temperature=0.0 保证输出尽量确定；seed=123 进一步固定随机性，便于结果复现
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0,
        seed=123,
    )

    # Return the model's response
    return response.choices[0].message.content


# 简单冒烟测试：确认 API Key 和网络连接可用，模型应回复 "hello world"
prompt = "Respond with 'hello world' if you got this message."
run_chatgpt(prompt, client)

In [ ]:
# 读取第 7 章主代码目录下已生成的原始指令微调数据集 instruction-data.json，作为本次“反思式改进”的输入数据
from pathlib import Path


json_file = Path("..") / "01_main-chapter-code" / "instruction-data.json"

with open(json_file, "r") as file:
    json_data = json.load(file)

# 打印数据条目总数，确认数据加载成功
print("Number of entries:", len(json_data))

In [ ]:
# 使用 pprint 美化打印，查看数据集第一条样本的结构（包含 instruction / input / output 字段）
from pprint import pp as pprint

pprint(json_data[0])

In [ ]:
# 构造“指令反思”提示词（Reflection-Tuning 第一阶段：针对原始 instruction+output 生成更复杂、更难的新指令）
# 适用于 input 字段为空的样本
def build_instruction_reflection_prompt_no_input(ins, outp):

    # system prompt：要求模型扮演一个“挑剔”的质量审查助手
    sys_prompt = "You are a helpful, precise but picky assistant for checking the quality of a given instruction."
    prompt_template = "[Instruction]\n{ins}\n\n[The Start of Answer]\n{outp}\n\n[The End of Answer]\n\n[System]\n{criteria}\n\n"
    # criteria：分三步要求模型 1) 分析原指令/原答案为何不够好  2) 据此生成一条更复杂、独立可答的新指令  3) 给出新指令对应的新答案
    criteria = "We would like you to answer several questions related to the quality of a given instruction. \n" + \
                "1. Why this instruction is not good? First analyse the instruction based on Complexity of the Topic, Level of Detail Required, Knowledge Required, Ambiguity of the Instruction and Logical Reasoning or Problem-Solving Involved. \n" + \
                "Then analyse why this answer is not good for the given instruction? Analyse based on the Helpfulness, Relevance, Accuracy and Level of Details. \n" + \
                "Finally analyse why this bad instruction lead to a bad answer. " +\
                "2. Based on the reason you provided, generate a new and complete instruction which is complex and difficult to answer directly. " + \
                "Make sure the new instruction is relevent but independent to the original instruction, which can be answered without knowing the original instruction, put the new instruction in the format of [New Instruction] your instruction [End]" +\
                "3. Answer the newly generated instruction as detailed as possible, in the format of [New Answer] your answer [End] \n"
    # 将原始 instruction(ins)、output(outp) 以及上面的评价标准(criteria)填入模板，拼出最终发给模型的 prompt
    prompt = prompt_template.format(
        ins=ins, outp=outp, criteria=criteria
    )
    return sys_prompt, prompt

In [ ]:
# 查看数据集第 3 条（索引为 2）样本原文，作为后续反思流程的演示样例
print(json_data[2])

In [ ]:
# 用上面构造的“指令反思”提示词，对示例样本调用 GPT，观察模型给出的分析 + 新指令 + 新答案
entry = json_data[2]

system_prompt, prompt = build_instruction_reflection_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)

print(output)

In [ ]:
# 定义一组正则解析函数，用于从模型返回的自由文本中提取出 [New Instruction]...[End] 和 [New Answer]...[End] 之间的内容
import re

# 提取“新指令”片段：优先匹配 [New Instruction] 标签，找不到则退化匹配 "New Instruction:" 文本形式
def extract_instruction_segment(text, no_input=True):
    if '[New Instruction]' in text:
        pattern = r'(\[New Instruction\])(.*?)(\[End\]|\[New Answer\]|New Answer:)'
    else:
        pattern = r'(New Instruction:)(.*?)(\[End\]|\[New Answer\]|New Answer:)'
    segments = re.findall(pattern, text, re.DOTALL)
    if len(segments) == 0:
        seg_ins = ''
    else:
        seg_ins = segments[0][1].strip()
    # 兼容模型偶尔把下一段编号 "3." 也带进来的情况，做一次截断清理
    if seg_ins.endswith("\n\n3."):
        seg_ins = seg_ins[:-4]
    return seg_ins


# 提取“新答案”片段：优先匹配 [New Answer] 标签，找不到则退化匹配 "New Answer:" 文本形式
def extract_output_segment(text, no_input=True):
    if '[New Answer]' in text:
        pattern = r'(\[New Answer\])(.*?)(\[End\]|$)'
    else:
        pattern = r'(New Answer:)(.*?)(\[End\]|$)'
        # pattern = r'(\[New Answer\]|New Answer:)(.*?)(\[End\]|$)'
    segments = re.findall(pattern, text, re.DOTALL)
    if len(segments) == 0:
        seg_oup = ''
    else:
        seg_oup = segments[0][1].strip()
    return seg_oup


# 汇总函数：同时提取新指令与新答案，若模型无输出（空字符串）则直接返回空列表
def extract_instruction(text):
    if text == '':
        return []
    seg_ins = extract_instruction_segment(text, no_input=True)
    seg_oup = extract_output_segment(text, no_input=True)
    return [seg_ins, seg_oup]

In [ ]:
# 对上一步的模型输出做解析，拆出新指令 new_instr 与新答案 new_outp
new_instr, new_outp = extract_instruction(output)

In [ ]:
# 打印解析出的新指令，检查提取效果
print(new_instr)

In [ ]:
# 打印解析出的新答案，检查提取效果
print(new_outp)

In [ ]:
# Reflection-Tuning 第二阶段：针对（可能已被替换过的）instruction 和 output，构造“答案反思”提示词，
# 要求模型评价现有答案质量并生成一个更好的答案。分别提供“无 input”与“有 input”两个版本
def build_response_reflection_prompt_no_input(ins, outp):

    sys_prompt = "You are a helpful, precise but picky assistant for checking the quality of the answer to a given instruction."
    prompt_template = "[Instruction]\n{ins}\n\n[The Start of Answer]\n{outp}\n\n[The End of Answer]\n\n[System]\n{criteria}\n\n"
    # criteria：1) 分析当前答案为何不够好  2) 生成一个更详细、更完整的新答案，用 [Better Answer]...[End] 包裹
    criteria = "We would like you to answer several questions related to the quality of the answer to the given instruction. \n" + \
                "1. Why this answer is not good for the given instruction? Analyse based on the Helpfulness, Relevance, Accuracy and Level of Details. \n" + \
                "2. Based on the reason you provided, generate a better answer, new and complete, as detailed as possible, in the format of [Better Answer] your answer [End] \n"
    prompt = prompt_template.format(
        ins=ins, outp=outp, criteria=criteria
    )
    return sys_prompt, prompt


# 与上面类似，但额外携带 input 字段（用于那些指令附带输入上下文的样本）
def build_response_reflection_prompt_with_input(ins, inp, outp):

    sys_prompt = "You are a helpful and precise assistant for checking the quality of the answer to a given instruction and its input."
    prompt_template = "[Instruction]\n{ins}\n\n[The Start of Input]\n{inp}\n\n[The End of Input]\n\n[The Start of Answer]\n{outp}\n\n[The End of Answer]\n\n[System]\n{criteria}\n\n"
    criteria = "We would like you to answer several questions related to the quality of the answer to the given instruction and corresponding input. \n" + \
                "1. Why this answer is not good for the given instruction and corresponding input? Analyse based on the Helpfulness, Relevance, Accuracy and Level of Details. \n" + \
                "2. Based on the reason you provided, generate a better answer, new and complete, as detailed as possible, in the format of [Better Answer] your answer [End] \n"
    prompt = prompt_template.format(
        ins=ins, inp=inp, outp=outp, criteria=criteria
    )
    return sys_prompt, prompt

In [ ]:
# 用“答案反思”提示词（无 input 版本）对示例样本再次调用 GPT，观察模型对答案质量的评价与改进后的新答案
entry = json_data[2]

system_prompt, prompt = build_response_reflection_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)

print(output)

In [ ]:
# 从模型输出中解析出 [Better Answer]...[End] 标签内的内容；若文本中出现两次及以上 [Better Answer]，
# 说明模型可能重复输出了多段，用不同的正则边界分别处理，最终返回所有匹配到的“更好答案”列表
def extract_response(text):
    if text.count('[Better Answer]') >= 2:
        pattern = r'\[(Better Answer)\](.*?)(\[End\]|\[Better Answer\]|$)'
        segments = re.findall(pattern, text, re.DOTALL)
    else:
        # pattern = r'\[(Better Answer)\](.*?)\[End\]'
        pattern = r'\[(Better Answer)\](.*?)(\[End\]|End|$)'
        segments = re.findall(pattern, text, re.DOTALL)
    return [segment[1].strip() for segment in segments]

In [ ]:
# 取解析结果中的第一段“更好答案”并打印，验证提取效果
response = extract_response(output)[0]
print(response)

In [ ]:
# 演示阶段：仅取数据集前 3 条样本用于小规模测试，避免消耗过多 API 调用额度
data_to_process = json_data[:3]

In [ ]:
# 批量执行“指令反思”流程（Reflection-Tuning 第一阶段）：
# 对每条 input 为空的样本，用 GPT 生成更复杂的新指令+新答案并替换原样本；
# 有 input 的样本本流程不处理新指令生成，直接原样保留
from tqdm import tqdm


def reflect_instructions(json_data, client):
    new_json_data = []

    # tqdm 包裹循环以显示处理进度条
    for entry in tqdm(json_data):

        if not entry["input"]:
            system_prompt, prompt = build_instruction_reflection_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
            output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)
            new_instr, new_outp = extract_instruction(output)
            new_entry = {"instruction": new_instr, "input": "", "output": new_outp}
            new_json_data.append(new_entry)
        else:
            # input 非空的样本暂不做指令级反思改写，直接保留原样本
            new_json_data.append(entry)

    return new_json_data
# 重新取前 3 条样本作为本次批量处理的输入
data_to_process = json_data[:3]

# 对这 3 条样本执行指令反思，得到改写后的新数据集
new_json_data = reflect_instructions(data_to_process, client)

In [ ]:
# 打印反思改写后的前 3 条新样本，人工检查生成质量
for i in new_json_data[:3]:
    pprint(i)
    print("\n\n")

In [ ]:
# 将“指令反思”阶段生成的新数据集保存为 instruction-reflected.json，供后续步骤或人工检查使用
with open("instruction-reflected.json", "w") as file:
    json.dump(new_json_data, file, indent=4)

In [ ]:
# 进入“答案反思”阶段：重新取原始数据集（而非上一步已被指令改写的数据）前 3 条样本作为待处理数据
data_to_process = json_data[:3]

In [ ]:
# 批量执行“答案反思”流程（Reflection-Tuning 第二阶段）：
# 对每条样本调用 GPT 生成更好的答案，并用新答案替换原 output；instruction/input 保持不变
def reflect_responses(json_data, client):
    new_json_data = []

    for entry in tqdm(json_data):

        if not entry["input"]:
            # input 为空：使用无 input 版本的提示词模板
            system_prompt, prompt = build_response_reflection_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
            output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)
            new_response = extract_response(output)

            # 若正则未能从模型输出中解析出任何 [Better Answer] 片段，则回退使用原始 output，避免数据丢失
            if not len(new_response):
                new_response = entry["output"]

            new_entry = {"instruction": entry["instruction"], "input": "", "output": new_response[0]}
            new_json_data.append(new_entry)

        else:
            # input 非空：使用带 input 版本的提示词模板
            system_prompt, prompt = build_response_reflection_prompt_with_input(ins=entry["instruction"], inp=entry["input"], outp=entry["output"])
            output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)
            new_response = extract_response(output)

            # 同样地，解析失败时回退使用原始 output
            if not len(new_response):
                new_response = entry["output"]

            new_entry = {"instruction": entry["instruction"], "input": entry["input"], "output": new_response[0]}
            new_json_data.append(new_entry)

    return new_json_data

In [ ]:
# 对前 3 条样本执行答案反思，得到答案质量提升后的新数据集
new_json_data = reflect_responses(data_to_process, client)

In [ ]:
# 打印答案反思改写后的前 3 条新样本，人工检查生成质量
for i in new_json_data[:3]:
    pprint(i)
    print("\n\n")

In [ ]:
# 将“答案反思”阶段生成的新数据集保存为 response-reflected.json，作为本 notebook 的最终产出
with open("response-reflected.json", "w") as file:
    json.dump(new_json_data, file, indent=4)